# 00 — Setup

Run this notebook **once on every new machine** (a Vast.ai server, your own computer, ...).
Run the cells from top to bottom.

If the repository is not on the machine yet, open a terminal in Jupyter, clone it, and then open this notebook
from the `stark-cls-finetune/notebooks/` folder:
```bash
git clone https://github.com/balk21/stark-cls-finetune.git
```

| Step | What it does | Time (approx.) |
|---|---|---|
| 1 | Creates the `vot1` conda environment (if missing) | 10–20 min |
| 2 | Checks the environment, the GPU and the paths | seconds |
| 3 | Downloads the STARK checkpoint (ST101 ≈ 180 MB) | 1 min |
| 4 | Downloads the VOT-LT2020 dataset (50 sequences, ≈ 17 GB) | 20–60 min |
| 5 | Smoke test: loads the model and tracks a few frames | 30 s |

**Notebook kernel:** any Python 3 kernel works (Jupyter's default). The actual work runs in the background with
the Python of the `vot1` environment, so you do not need to change the kernel.

**GPU:** the environment ships PyTorch 2.4.1 + CUDA 12.1. Tested on Vast.ai with **RTX 3000 / 4000 series** GPUs.
RTX 5000 series (Blackwell) GPUs do not work with this PyTorch version.

In [ ]:
# Run this cell once at the top of every notebook.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebooks/ folder (nbhelper.py lives here)
import nbhelper as nb

In [ ]:
print("Repository root:", nb.REPO_ROOT)
print("conda          :", nb.find_conda())
print("vot1           :", nb.find_env_python())

## 1. The `vot1` environment

Does nothing if the environment already exists. If `conda` is not found, pick an image that ships conda
(on Vast.ai, a PyTorch template with conda) or install conda first.

In [ ]:
if nb.find_env_python() is None:
    conda = nb.find_conda()
    assert conda, "conda not found: use an image with conda or install Miniconda."
    nb.stream([conda, "env", "create", "-n", nb.ENV_NAME, "-f", nb.REPO_ROOT / "environment" / "vot1_environment.yml"])
    print("\nCreated environment:", nb.find_env_python())
else:
    print("The vot1 environment already exists:", nb.find_env_python())

## 2. Environment check

Prints package versions, the GPU and the paths that will be used. If a **PROBLEMS** section appears at the bottom,
fix those first.

By default all paths are inside the repository (`checkpoints/`, `data/votlt2020/sequences/`, `outputs/`).
To use other locations, see *Optional: different paths* at the end of this notebook.

In [ ]:
nb.cli("check", check=False)

## 3. Download checkpoints

The official STARK checkpoints are downloaded from Google Drive into
`checkpoints/<stark_st2|stark_s>/<model_config>/`.

| `MODEL` | values for `MODEL_CONFIGS` | Description |
|---|---|---|
| `stark_st` | `baseline_R101` | **STARK-ST101** (default, used in the experiments) |
| `stark_st` | `baseline` | STARK-ST50 |
| `stark_st` | `baseline_R101_got10k_only`, `baseline_got10k_only` | versions trained on GOT-10k only |
| `stark_s` | `baseline`, `baseline_got10k_only` | STARK-S50 (no confidence score, cannot be fine-tuned) |

If Drive reports "quota exceeded", download the file in a browser from the link in the error message and put it
in the folder shown there.

**Your own checkpoint** (e.g. `STARKSTcoco_ep0050.pth.tar`): upload it to
`checkpoints/stark_st2/baseline_R101/` and set `checkpoint="STARKSTcoco_ep0050.pth.tar"` in the experiment.

In [ ]:
MODEL = "stark_st"
MODEL_CONFIGS = ["baseline_R101"]          # e.g. ["baseline_R101", "baseline"]

nb.cli("download-checkpoints", "--model", MODEL, "--model-config", *MODEL_CONFIGS)

## 4. VOT-LT2020 dataset

Downloaded with vot-toolkit (≈ 17 GB, 50 sequences). Make sure at least **25 GB** of disk space is free.
Nothing is downloaded if the dataset is already present. If the download is interrupted, run the cell again.

In [ ]:
import shutil
free_gb = shutil.disk_usage(nb.REPO_ROOT).free / 1e9
print(f"Free disk space: {free_gb:.1f} GB")
nb.cli("download-dataset")

## 5. Smoke test

Loads the model without vot-toolkit and tracks the first 50 frames of a sequence.
A mean IoU above 0.5 and a reasonable FPS (~20–30 on an RTX 3060) mean the setup is correct.

In [ ]:
nb.cli("smoke", "--frames", "50", "--set", "ft_mode=online")

Setup complete. Open **`01_run_experiment.ipynb`** to run experiments.

---
## Optional: different paths

To keep checkpoints, the dataset or the outputs on another disk, edit and run the cell below.
It creates `configs/paths.local.yaml` (ignored by git). Keep only the lines you want to change.

In [ ]:
LOCAL_PATHS = '''
# checkpoints: /workspace/stark_checkpoints
# dataset: /workspace/votlt2020/sequences
# outputs: /workspace/stark_outputs
'''
WRITE = False   # set to True and run
if WRITE:
    (nb.REPO_ROOT / "configs" / "paths.local.yaml").write_text(LOCAL_PATHS.strip() + "\n")
    nb._PATHS_CACHE.clear()
    print(nb.paths())

## Optional: registering `vot1` as a Jupyter kernel

**Not required.** Only useful if you want to import the `stark_ft` modules directly in a notebook and write your
own analyses (adds the `ipykernel` package to the `vot1` environment). Then choose *Kernel → Change kernel →
Python (vot1)*.

In [ ]:
REGISTER_KERNEL = False   # set to True and run
if REGISTER_KERNEL:
    py = nb.python()
    nb.stream([py, "-m", "pip", "install", "-q", "ipykernel"])
    nb.stream([py, "-m", "ipykernel", "install", "--user", "--name", "vot1", "--display-name", "Python (vot1)"])